# Future forecasts (Figure 6)

Every method is re-fitted with jump-off year **2019**, the last year the data
supports, and forecasts forward. This notebook produces Figure 6 of the paper: the
mean completed cohort fertility (ages 15-40) each method forecasts, averaged over
the 26 countries that every method covers.

Inputs are written to `data/future_forecasts/` by `code/future_forecasts/run_all.py`.
Most of these cohorts are not yet complete in the data, so the figure shows what
the methods forecast, not how accurate they are. Accuracy is evaluated in
`eval_figures_cohort.ipynb`.

## Load forecasts

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Arial throughout, to match the other figures in the paper
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset": "stixsans",
    "pdf.fonttype": 42,   # embed TrueType fonts
    "ps.fonttype": 42,
})

DATA_DIR = '../../data/future_forecasts'
FIG_DIR = '../../figures'
FUTURE_JOY = 2019

# label -> forecast file, observed file, and colour (same colours as the
# other figures in the paper)
MODEL_REGISTRY = {
    'Lee':                dict(pred='lee2_forecasts_cohort',            obs='lee2_obs_cohort',              color='#d62728'),
    'CCF-Net':            dict(pred='dl_forecasts_future_joy2019',      obs='dl_obs_cohort_future_joy2019', color='#8c564b'),
    'Freeze rates':       dict(pred='freeze_forecasts_cohort',          obs='freeze_obs_cohort',            color='#2ca02c'),
    'Myrskyla et al.':    dict(pred='myrskyla_forecasts_cohort',        obs='myrskyla_obs_cohort',          color='#9467bd'),
    'de Beer':            dict(pred='debeer2_forecasts_cohort',         obs='debeer2_obs_cohort',           color='#ff7f0e'),
    'Schmertmann et al.': dict(pred='schmertmann2013_forecasts_cohort', obs='schmertmann2013_obs_cohort',   color='#7f7f7f'),
    'Sevcikova et al.':   dict(pred='sevcikova_forecasts_cohort',       obs='sevcikova_obs_cohort',         color='#e377c2'),
}
EMPHASIS = 'CCF-Net'      # drawn heavier

# Completed fertility = sum of ASFR over ages 15-40, as in Bohk-Ewald et al. (2018)
CF_AGE_LO, CF_AGE_HI = 15, 40
N_CF_AGES = CF_AGE_HI - CF_AGE_LO + 1

# geo index -> HFD country code
GEO_NAMES = {int(idx): str(name)
             for name, idx in np.load('../../data/geos_key.npy', allow_pickle=True)}

M = {}
for label, meta in MODEL_REGISTRY.items():
    M[label] = {k: pd.read_csv(f"{DATA_DIR}/{meta[k]}.csv") for k in ('pred', 'obs')}
    for k in ('pred', 'obs'):
        # Country is quoted in some of the R-written files and bare in others
        M[label][k]['Country'] = M[label][k]['Country'].astype(int)

## Completed cohort fertility at jump-off 2019

In [ ]:
# Completed cohort fertility (CF) implied by each method at the jump-off year.
# A cohort's ages up to its age in 2019 are observed and the rest are forecast,
# so CF = sum of observed rates (ages 15..AAF) + forecast rates (ages AAF+1..40),
# where AAF = 2019 - birth cohort. Cohorts missing any age are dropped.
def spliced_cf(pred_df, obs_df, joy=FUTURE_JOY):
    obs = (obs_df[obs_df['Age'].between(CF_AGE_LO, CF_AGE_HI)]
           [['Country', 'Year', 'Age', 'Rate']]
           .drop_duplicates(['Country', 'Year', 'Age']))
    obs_hist = obs[obs['Year'] + obs['Age'] <= joy]
    fcast = pred_df[pred_df['JumpOffYear'] == joy][['Country', 'Year', 'Age', 'Rate']]
    fcast = fcast[(fcast['Year'] + fcast['Age'] > joy) &
                  fcast['Age'].between(CF_AGE_LO, CF_AGE_HI)]
    traj = pd.concat([obs_hist, fcast])
    n_ages = traj.groupby(['Country', 'Year'])['Age'].transform('nunique')
    res = (traj[n_ages == N_CF_AGES].groupby(['Country', 'Year'])['Rate']
           .sum().reset_index(name='cf'))
    res['AAF'] = joy - res['Year']
    return res


cf_level = {label: spliced_cf(m['pred'], m['obs']) for label, m in M.items()}

# Observed CF for every cohort that is complete in the data. The observed
# histories are the same in every method's file, so CCF-Net's is used.
_obs = (M['CCF-Net']['obs'].query('@CF_AGE_LO <= Age <= @CF_AGE_HI')
        [['Country', 'Year', 'Age', 'Rate']].drop_duplicates(['Country', 'Year', 'Age']))
_n_ages = _obs.groupby(['Country', 'Year'])['Age'].transform('nunique')
obs_cf = (_obs[_n_ages == N_CF_AGES].groupby(['Country', 'Year'])['Rate']
          .sum().rename('cf').reset_index())

## Country sample and means

In [ ]:
# Country sample and cross-country means.
# Forecast cohorts are those aged 15-39 in 2019 (born 1980-2004). A country is
# kept only if:
#   1. every method forecasts CF for all of those cohorts, and
#   2. every method's CF stays within CF_BAND over the whole window. This drops
#      Taiwan, where de Beer's forecast diverges (CF from -579 to +362).
# This gives the 26 countries in Figure 6. All lines are averaged over the same
# countries.
AAF_LO, AAF_HI = 15, 39
CF_BAND = (0.5, 3.5)       # plausible CF range, children per woman
LEAD_IN = 12               # observed cohorts drawn before the first forecast cohort
COHORT_HI = 1995           # last cohort drawn (aged 24 in 2019)

cohort_lo = FUTURE_JOY - AAF_HI        # 1980
n_cohorts = AAF_HI - AAF_LO + 1
fc = {label: d.query('@AAF_LO <= AAF <= @AAF_HI') for label, d in cf_level.items()}

full = set.intersection(*[
    set(g for g, n in d.groupby('Country').size().items() if n == n_cohorts)
    for d in fc.values()])
off_band = {g for d in fc.values()
            for g in d.loc[~d['cf'].between(*CF_BAND), 'Country']}
geos = sorted(full - off_band, key=lambda g: GEO_NAMES[g])

# mean forecast CF per cohort, per method
fc_mean = pd.DataFrame({
    label: d[d['Country'].isin(geos)].groupby('Year')['cf'].mean()
    for label, d in fc.items()}).loc[:COHORT_HI]

# observed mean over the same countries, for cohorts all of them have completed
o = obs_cf[obs_cf['Country'].isin(geos) &
           obs_cf['Year'].between(cohort_lo - LEAD_IN, COHORT_HI)]
o = o[o.groupby('Year')['Country'].transform('size') == len(geos)]
obs_mean = o.groupby('Year')['cf'].mean()

print(f'{len(geos)} countries: ' + ', '.join(GEO_NAMES[g] for g in geos))
print(f'Dropped for leaving {CF_BAND}: '
      + ', '.join(sorted(GEO_NAMES[g] for g in off_band & full)))
print(f'Observed mean CF, cohort {obs_mean.index[-1]}: {obs_mean.iloc[-1]:.3f}')
print(f'Mean forecast CF, cohort {COHORT_HI}:')
print(fc_mean.iloc[-1].sort_values().round(3).to_string())

## Figure 6: mean completed fertility forecasts

In [ ]:
# Figure 6
LABEL_GAP = 0.055     # min vertical spacing of the right-edge labels
FS_LAB = 8.5
aaf_end = FUTURE_JOY - COHORT_HI

fig, ax = plt.subplots(figsize=(7.0, 4.2), layout='constrained')

# shaded: cohorts not yet complete in the data
ax.axvspan(obs_mean.index[-1], COHORT_HI + 0.4, color='0.92', lw=0, zorder=0)
ax.plot(obs_mean.index, obs_mean.values, color='black', lw=2.2, zorder=6)

for label in fc_mean:
    em = label == EMPHASIS
    ax.plot(fc_mean.index, fc_mean[label], color=MODEL_REGISTRY[label]['color'],
            lw=2.4 if em else 1.2, zorder=5 if em else 3)

# right-edge labels instead of a legend, pushed apart where values are close
end = fc_mean.iloc[-1].sort_values(ascending=False)
label_y, prev = {}, np.inf
for label, v in end.items():
    label_y[label] = min(v, prev - LABEL_GAP)
    prev = label_y[label]
for label, v in end.items():
    color = MODEL_REGISTRY[label]['color']
    ax.annotate(f'{label}  {v:.2f}', xy=(COHORT_HI, v),
                xytext=(COHORT_HI + 1.2, label_y[label]),
                color=color, fontsize=FS_LAB,
                fontweight='bold' if label == EMPHASIS else 'normal',
                va='center', ha='left', annotation_clip=False,
                arrowprops=dict(arrowstyle='-', color=color, lw=0.7,
                                shrinkA=0, shrinkB=1, alpha=0.7))
ax.annotate(f'Observed  {obs_mean.iloc[-1]:.2f}',
            xy=(obs_mean.index[-1], obs_mean.iloc[-1]),
            xytext=(obs_mean.index[-1] - 0.8, obs_mean.iloc[-1] + 0.045),
            color='black', fontsize=FS_LAB, va='bottom', ha='right')

ax.set_xlim(obs_mean.index[0], COHORT_HI + 0.4)
span = float(np.nanmax(fc_mean.values) - np.nanmin(fc_mean.values))
ax.set_ylim(float(np.nanmin(fc_mean.values)) - 0.06 * span, None)
ax.set_xlabel('Birth cohort')
ax.set_ylabel(f'Mean completed cohort fertility, ages {CF_AGE_LO}-{CF_AGE_HI}\n'
              f'({len(geos)} countries, children per woman)', linespacing=1.5)
ax.grid(True, ls=':', alpha=0.5)
ax.set_axisbelow(True)
ax.text(0.012, 0.975,
        f'Shaded: cohorts not yet complete in the data.\nCohort {COHORT_HI} was aged '
        f'{aaf_end} at the jump-off, so {CF_AGE_HI - aaf_end} of its '
        f'{N_CF_AGES} childbearing ages are forecast.',
        transform=ax.transAxes, fontsize=7.5, color='0.4', va='top', ha='left',
        linespacing=1.6)

os.makedirs(FIG_DIR, exist_ok=True)
png = f'{FIG_DIR}/cf_mean_future_joy{FUTURE_JOY}.png'
fig.savefig(png, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {png}')